In [ ]:
import json
import shutil
from datetime import timedelta
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
from shapely.geometry import shape, Point
from shapely.prepared import prep
import shapely.vectorized

import parcels
from parcels import StatusCode


In [ ]:
YEAR = 2025
integration_days = 5
integration_direction = 1
dt_minutes = 5

max_sweep_dates = None

refine_lon_fac = 5
refine_lat_fac = 5

fname_hourly = "../data/hourly_glorys/cabo_verde_TUV_20_25_hourly_glorys_landmasked.nc"
fname_edge = "../data/hourly_glorys/cabo_verde_TUV_20_25_hourly_glorys_landmasked_zero_edges.nc"
output_dir = "../data/dispersal_and_connectivity_tests"
geojson_path = "../data/island_buffers_nonoverlap.geojson"
island_name_field = "ISLAND"


In [ ]:
release_dates = [t.strftime("%Y-%m-%d")
                 for t in pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq=f"{integration_days}D")]
if max_sweep_dates is not None:
    release_dates = release_dates[:max_sweep_dates]
print(f"{len(release_dates)} release dates for the {integration_days}-day dispersal sweep in {YEAR}")


In [ ]:
ds = xr.open_dataset(fname_hourly).squeeze()
lon_f, lat_f = ds["lon_f"], ds["lat_f"]

if not Path(fname_edge).exists():
    raise FileNotFoundError(
        f"{fname_edge} doesn't exist -- run ftle_calc_and_plots.ipynb first, "
        f"it builds this file once, shared by both notebooks."
    )

filenames = {"U": fname_edge, "V": fname_edge}
variables = {"U": "sozocrtx", "V": "somecrty"}
dimensions = {
    "U": {"lon": "lon_f", "lat": "lat_f", "time": "time_counter"},
    "V": {"lon": "lon_f", "lat": "lat_f", "time": "time_counter"},
}
fieldset = parcels.FieldSet.from_nemo(
    filenames, variables, dimensions, allow_time_extrapolation=True, deferred_load=False,
)


In [ ]:
lon_min, lon_max = float(lon_f.min()), float(lon_f.max())
lat_min, lat_max = float(lat_f.min()), float(lat_f.max())

plon = xr.DataArray(np.linspace(lon_min, lon_max, refine_lon_fac * ds.sizes["x"] - 1), name="plon", dims="plon")
plat = xr.DataArray(np.linspace(lat_min, lat_max, refine_lat_fac * ds.sizes["y"] - 1), name="plat", dims="plat")
plon, plat = xr.broadcast(plon, plat)

cand_lon = plon.stack(pid=["plon", "plat"]).load().data
cand_lat = plat.stack(pid=["plon", "plat"]).load().data
print(f"{cand_lon.size} candidate release-grid particles")


In [ ]:
with open(geojson_path) as f:
    gj = json.load(f)

islands = [{"name": feat["properties"][island_name_field], "geom": shape(feat["geometry"])}
           for feat in gj["features"]]
for isl in islands:
    isl["prepared"] = prep(isl["geom"])
n_islands = len(islands)

source_id = np.full(cand_lon.shape, -1, dtype=np.int32)
for fid, isl in enumerate(islands):
    minx, miny, maxx, maxy = isl["geom"].bounds
    in_bbox = (cand_lon >= minx) & (cand_lon <= maxx) & (cand_lat >= miny) & (cand_lat <= maxy)
    candidates = np.where(in_bbox)[0]
    inside = np.array([isl["prepared"].contains(Point(cand_lon[i], cand_lat[i])) for i in candidates])
    source_id[candidates[inside]] = fid

keep = source_id >= 0
particle_lon = cand_lon[keep]
particle_lat = cand_lat[keep]
source_id = source_id[keep]
particle_ids = np.arange(particle_lon.size)

island_names = [isl["name"] for isl in islands]
dest_labels = island_names + ["open_ocean"]
print(f"{particle_lon.size} / {cand_lon.size} candidate particles fall inside an island buffer")


In [ ]:
class IslandParticle(parcels.JITParticle):
    pid_orig = parcels.Variable("pid_orig", dtype=np.int32)
    source_id = parcels.Variable("source_id", dtype=np.int32, to_write="once")


def check_out_of_bounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        particle.delete()


def check_error(particle, fieldset, time):
    if particle.state >= 50:
        particle.delete()


In [ ]:
output_dir_path = Path(output_dir)
output_dir_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def run_release(release_date):
    release_tag = pd.Timestamp(release_date).strftime("%Y%m%d")
    traj_path = output_dir_path / f"island_connectivity_{integration_days}d_{release_tag}.zarr"
    if traj_path.exists():
        shutil.rmtree(traj_path)

    pset = parcels.ParticleSet.from_list(
        fieldset=fieldset, pclass=IslandParticle,
        lon=particle_lon, lat=particle_lat,
        time=np.datetime64(pd.Timestamp(release_date)),
        pid_orig=particle_ids, source_id=source_id,
    )
    output_file = pset.ParticleFile(name=str(traj_path), outputdt=timedelta(hours=1))
    kernels = pset.Kernel(parcels.AdvectionRK4) + pset.Kernel(check_out_of_bounds) + pset.Kernel(check_error)
    pset.execute(
        kernels,
        runtime=timedelta(days=integration_days),
        dt=timedelta(minutes=int(integration_direction * dt_minutes)),
        output_file=output_file,
    )

    # final position per particle, ignoring any particle stuck at its release point
    ds_out = xr.open_dataset(traj_path, engine="zarr")
    stuck = ((ds_out.lon.diff("obs").isel(obs=0) == 0) &
             (ds_out.lat.diff("obs").isel(obs=0) == 0)).squeeze().compute()
    ds_clean = ds_out.isel(trajectory=~stuck.values)

    lon_vals, lat_vals = ds_clean["lon"].values, ds_clean["lat"].values
    valid = np.isfinite(lon_vals)
    last_idx = valid.shape[1] - 1 - np.argmax(valid[:, ::-1], axis=1)
    traj_idx = np.arange(lon_vals.shape[0])
    final_lon, final_lat = lon_vals[traj_idx, last_idx], lat_vals[traj_idx, last_idx]

    src = ds_clean["source_id"].values
    src = src[:, 0] if src.ndim > 1 else src

    dest = np.full(final_lon.shape, -1, dtype=int)
    for fid, isl in enumerate(islands):
        dest[shapely.vectorized.contains(isl["geom"], final_lon, final_lat)] = fid

    connectivity = np.zeros((n_islands, n_islands + 1), dtype=int)
    for s in range(n_islands):
        d = dest[src == s]
        d[d == -1] = n_islands  # open_ocean is the last column
        connectivity[s] = np.bincount(d, minlength=n_islands + 1)

    conn_path = output_dir_path / f"connectivity_{integration_days}d_{release_tag}.npz"
    np.savez_compressed(conn_path, connectivity=connectivity)
    return connectivity, conn_path


In [ ]:
conn_csv_path = output_dir_path / f"island_connectivity_matrix_{integration_days}d_{YEAR}.csv"

if conn_csv_path.exists():
    print(f"Found {conn_csv_path}, reusing it, skipping the sweep.")
    connectivity_total = pd.read_csv(conn_csv_path, index_col="source_island").values
else:
    connectivity_total = np.zeros((n_islands, n_islands + 1), dtype=int)
    for release_date in release_dates:
        release_tag = pd.Timestamp(release_date).strftime("%Y%m%d")
        conn_path = output_dir_path / f"connectivity_{integration_days}d_{release_tag}.npz"

        if conn_path.exists():
            connectivity_total += np.load(conn_path)["connectivity"]
            continue

        print(f"Running {integration_days}-day release from {release_date} ...")
        connectivity, _ = run_release(release_date)
        connectivity_total += connectivity


In [ ]:
conn_df = pd.DataFrame(connectivity_total, index=island_names, columns=dest_labels)
conn_df.index.name = "source_island"
conn_df.columns.name = f"destination_island (after {integration_days} days, summed over {YEAR})"
conn_df.to_csv(conn_csv_path)
conn_df
